# imports

In [16]:
import numpy as np
import pandas as pd
import hypertools as hyp
import numpy as np
import os
from scipy import ndimage
from scipy.spatial.distance import cdist
from scipy.signal import resample
from scipy.stats import zscore
from scipy.spatial.distance import correlation
from scipy.interpolate import interp1d as interpolate
%matplotlib inline

# paths to data dirs

In [17]:
annot_dir = '../../data/annotations_dfs/'
model_dir = '../../data/models/'

# load annotations dataframes

In [3]:
atlep1_df = pd.read_pickle(annot_dir+'atlep1.p')
atlep2_df = pd.read_pickle(annot_dir+'atlep2.p')
arrdev_df = pd.read_pickle(annot_dir+'arrdev.p')

# model parameters

In [4]:
n_topics = 100
episode_wsize = 50
recall_wsize = 10

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

# fit model to episode annotations, get sliding windows for modeling recalls

In [5]:
def fit_episode_model(episode_df, n_topics, episode_wsize, vec_params, sem_params):
    
    # throw all annotations into bag of words to train model
    episode_bag = episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(lambda x: ', '.join(x.fillna('')), axis=1).values.tolist()
    
    # create list for annotation sliding windows (of size w_size)
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(','.join(episode_bag[idx:idx+episode_wsize]))
        
    # use hypertools to create episode model
    return {'model': hyp.tools.format_data(episode_w, vectorizer=vec_params, semantic=sem_params, 
                                 corpus=episode_w)[0], 'windows': episode_w}

In [6]:
atlep1_model_ws = fit_episode_model(atlep1_df, n_topics, episode_wsize, vectorizer_params, semantic_params)
atlep1_model, atlep1_windows = atlep1_model_ws['model'], atlep1_model_ws['windows']

atlep2_model_ws = fit_episode_model(atlep2_df, n_topics, episode_wsize, vectorizer_params, semantic_params)
atlep2_model, atlep2_windows = atlep2_model_ws['model'], atlep2_model_ws['windows']

arrdev_model_ws = fit_episode_model(arrdev_df, n_topics, episode_wsize, vectorizer_params, semantic_params)
arrdev_model, arrdev_windows = arrdev_model_ws['model'], arrdev_model_ws['windows']

## interpolation

In [7]:
endframe_times = {'atlep1_df': 1466.0, 'atlep2_df': 1316.52, 'arrdev_df': 1236.6}

In [8]:
episode_info = {
    'atlep1' : {
        'endframe_time' : 1466.0,
        'df' : atlep1_df,
        'model' : atlep1_model
    },
    'atlep2' : {
        'endframe_time' : 1316.52,
        'df' : atlep2_df,
        'model' : atlep2_model
    },
    'arrdev' : {
        'endframe_time' : 1236.6,
        'df' : arrdev_df,
        'model' : arrdev_model
    }
}

In [9]:
def find_midpoint_time(df, endframe_time):
    """
    returns list of timepoints at middle of each annotation segment
    """
    midpoint_times = []
    for i, tpt in enumerate(df['Onset time']):
        if i != len(df['Onset time'])-1:
            midpoint_time = np.mean([tpt, df['Onset time'][i+1]])
        else:
            midpoint_time = np.mean([tpt, endframe_time])
        midpoint_times.append(midpoint_time)

    return midpoint_times

In [10]:
def interpolate_model(episode_key, resolution):

    """
    resamples topic model timeseries to desired resolution. 'resolution' is in units of seconds
    """
    
    endframe_time = episode_key['endframe_time']
    df = episode_key['df']
    model = episode_key['model']
    
    # get middle timepoint for each annotation
    midpoint_times = find_midpoint_time(df, endframe_time)
    
    new_model = np.empty((int(round(endframe_time)),np.shape(model)[1]))
    
    # loop over topic dimensions
    for dim in range(np.shape(model)[1]):
        # values for given dimension at each timepoint
        single_dim = []
        for tpt in range(np.shape(model)[0]):
            single_dim.append(model[tpt][dim])
        
        # create interpolation function from dimension timeseries
        interp_func = interpolate(midpoint_times, single_dim, fill_value='extrapolate')
        
        # set of new timepoints
        new_tpts = np.arange(int(round(endframe_time)), step=resolution)
        
        # interpolate single topic dimension trajectory new timescale
        single_dim_res = interp_func(new_tpts)
        
        # fill in array for resampled model
        for ix, new_tpt in enumerate(single_dim_res):
            new_model[ix][dim] = new_tpt
    
    return new_model

In [11]:
atlep1_model_res = interpolate_model(episode_info['atlep1'], 1)
atlep2_model_res = interpolate_model(episode_info['atlep2'], 1)
arrdev_model_res = interpolate_model(episode_info['arrdev'], 1)

# save episode models

In [19]:
# create folder for models with parameters
if not os.path.isdir(model_dir+'video/t{0}_w{1}'.format(n_topics,episode_wsize)):
    os.mkdir(model_dir+'video/t{0}_w{1}'.format(n_topics,episode_wsize))

In [21]:
# save models
np.save(model_dir+'video/t{0}_w{1}/atlep1_model_t{0}_w{1}'.format(n_topics,episode_wsize), atlep1_model)
np.save(model_dir+'video/t{0}_w{1}/atlep2_model_t{0}_w{1}'.format(n_topics,episode_wsize), atlep2_model)
np.save(model_dir+'video/t{0}_w{1}/arrdev_model_t{0}_w{1}'.format(n_topics,episode_wsize), arrdev_model)

In [20]:
# save resampled models
np.save(model_dir+'video/t{0}_w{1}/atlep1_model_t{0}_w{1}_res'.format(n_topics,episode_wsize), atlep1_model_res)
np.save(model_dir+'video/t{0}_w{1}/atlep2_model_t{0}_w{1}_res'.format(n_topics,episode_wsize), atlep2_model_res)
np.save(model_dir+'video/t{0}_w{1}/arrdev_model_t{0}_w{1}_res'.format(n_topics,episode_wsize), arrdev_model_res)

# model recalls

In [12]:
def transform_recall(transcript, n_topics, recall_wsize, vec_params, sem_params, episode_windows, resample_shape):
    """
    transform recalls based on sliding windows of episode annotations
    """
    # sliding windows of episode annotations and recall words
    recall_w = []

    # split recall transcript into list
    rec_list = transcript.split()
    
    # create overlapping windows of recall_wsize words
    for ix, word in enumerate(rec_list):
        recall_w.append(','.join(rec_list[ix:ix+recall_wsize]))
        
    # create recall model
    recall_model = hyp.tools.format_data(recall_w, vectorizer=vec_params, semantic=sem_params,
                                         corpus=episode_windows)[0]
    
    # return recalled model to corresponding episode model length
    return resample(recall_model, resample_shape)

In [13]:
# load in and model each recall transcript

## save recall models

In [29]:
atlep1_model[12]

array([9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.99100817e-01, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
       9.08265213e-06, 9.08265213e-06, 9.08265213e-06, 9.08265213e-06,
      